# H12 - Confidence-Threshold Sweep / Calibration

This notebook addresses H12.1, H12.2, and H12.3.

- **H12.1:** Load H10 prediction/probability artifacts and sweep `(tau_low, tau_high)` pairs for the runtime 3-class verdict mapping.
- **H12.2:** Plot a reliability diagram for predicted scam probability vs observed scam frequency. If logits are available, fit a temperature-scaling parameter on the validation split first.
- **H12.3:** Save the selected thresholds, optional temperature parameter, decision markdown, and runtime-consumable JSON constants.

This notebook does **not** run model inference. It only consumes H10 CSV outputs from Google Drive.


## Install

The sweep is lightweight and runs on CPU.


In [ ]:
%pip install -q pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.1 matplotlib==3.9.0 scipy==1.13.1


## Drive Paths

H10 must write its prediction/probability CSV to Drive before this notebook can run.

Expected H10 artifact:

```text
/content/drive/MyDrive/GemScan/notebooks/_results/h10_baseline_predictions.csv
```

Expected schema:

```text
id,text,true_label,true_verdict,model_tier,safe_prob,suspicious_prob,scam_prob,predicted_verdict,source,split
```

`true_verdict` is preferred. `true_label` is accepted as a fallback. Optional logits columns can be included as `safe_logit,suspicious_logit,scam_logit` for temperature scaling.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

NOTEBOOKS_ROOT = Path("/content/drive/MyDrive/GemScan/notebooks")
DATA_DIR = NOTEBOOKS_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SCRUBBED_DIR = DATA_DIR / "scrubbed"
RESULTS_DIR = NOTEBOOKS_ROOT / "_results"
FIXTURES_DIR = DATA_DIR / "fixtures"
PROMPTS_DIR = DATA_DIR / "prompts"

for directory in [PROCESSED_DIR, SCRUBBED_DIR, RESULTS_DIR, FIXTURES_DIR, PROMPTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED_H10_ARTIFACT = RESULTS_DIR / "h10_baseline_predictions.csv"
H10_ARTIFACT_PATH = Path(os.environ.get("GEMSCAN_H10_PREDICTIONS", str(EXPECTED_H10_ARTIFACT)))

SWEEP_RESULTS_PATH = RESULTS_DIR / "h12_threshold_sweep.csv"
BEST_METRICS_PATH = RESULTS_DIR / "h12_best_threshold_metrics.json"
RELIABILITY_PLOT_PATH = RESULTS_DIR / "h12_reliability_diagram.png"
DECISION_SUMMARY_PATH = RESULTS_DIR / "h12_threshold_decision.md"
RUNTIME_CONSTANTS_PATH = RESULTS_DIR / "h12_runtime_threshold_constants.json"

LABELS = ["safe", "suspicious", "scam"]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

print("H10_ARTIFACT_PATH", H10_ARTIFACT_PATH)
print("RESULTS_DIR", RESULTS_DIR)


## Load H10 Predictions

This cell fails fast if H10 has not produced the required CSV. The failure message is intentionally explicit so H10 can be updated to write the handoff file.


In [ ]:
import numpy as np
import pandas as pd

REQUIRED_BASE_COLUMNS = {"id", "text", "model_tier", "safe_prob", "suspicious_prob", "scam_prob"}
OPTIONAL_CONTEXT_COLUMNS = ["predicted_verdict", "source", "split"]
OPTIONAL_LOGIT_COLUMNS = ["safe_logit", "suspicious_logit", "scam_logit"]

if not H10_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        "Missing H10 prediction/probability artifact. "
        f"Expected {H10_ARTIFACT_PATH}. "
        "H10 must produce a CSV with columns: "
        "id,text,true_label,true_verdict,model_tier,safe_prob,suspicious_prob,scam_prob,"
        "predicted_verdict,source,split. Optional temperature-scaling logits: "
        "safe_logit,suspicious_logit,scam_logit."
    )

predictions = pd.read_csv(H10_ARTIFACT_PATH)
missing = REQUIRED_BASE_COLUMNS - set(predictions.columns)
if missing:
    raise ValueError(f"H10 artifact is missing required columns: {sorted(missing)}")
if "true_verdict" not in predictions.columns and "true_label" not in predictions.columns:
    raise ValueError("H10 artifact must contain true_verdict or true_label.")

def normalize_label(value):
    raw = str(value).lower().strip()
    mapping = {
        "ham": "safe",
        "legitimate": "safe",
        "benign": "safe",
        "safe": "safe",
        "maybe": "suspicious",
        "ambiguous": "suspicious",
        "suspicious": "suspicious",
        "spam": "scam",
        "phishing": "scam",
        "fraud": "scam",
        "scam": "scam",
    }
    return mapping.get(raw)

label_source = "true_verdict" if "true_verdict" in predictions.columns else "true_label"
predictions["true_verdict_norm"] = predictions[label_source].map(normalize_label)
predictions = predictions.dropna(subset=["true_verdict_norm", "scam_prob", "safe_prob", "suspicious_prob"]).copy()
predictions["true_id"] = predictions["true_verdict_norm"].map(label2id)

prob_cols = ["safe_prob", "suspicious_prob", "scam_prob"]
for column in prob_cols:
    predictions[column] = pd.to_numeric(predictions[column], errors="coerce")
predictions = predictions.dropna(subset=prob_cols).copy()

prob_sum = predictions[prob_cols].sum(axis=1)
bad_prob_rows = predictions[(prob_sum <= 0) | (predictions[prob_cols] < 0).any(axis=1)]
if len(bad_prob_rows):
    raise ValueError(f"Found {len(bad_prob_rows)} rows with invalid probabilities.")
if not np.allclose(prob_sum, 1.0, atol=1e-3):
    predictions[prob_cols] = predictions[prob_cols].div(prob_sum, axis=0)
    print("Renormalized probability columns because row sums were not exactly 1.0.")

if "split" not in predictions.columns:
    predictions["split"] = "heldout"
predictions["split"] = predictions["split"].fillna("heldout").astype(str).str.lower().str.strip()

print("loaded rows", len(predictions))
print("columns", predictions.columns.tolist())
print("splits")
print(predictions["split"].value_counts())
print("labels")
print(predictions["true_verdict_norm"].value_counts())
predictions.head()


## Optional Temperature Scaling

If H10 writes logits, this cell fits a single temperature on the validation split and replaces the probability columns used by H12 with calibrated probabilities. If logits are absent, H12 continues with H10 probabilities and records that calibration was skipped.


In [ ]:
from scipy.optimize import minimize_scalar

def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)

def negative_log_likelihood(logits, labels, temperature):
    probs = softmax(logits / temperature)
    clipped = np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0)
    return -np.mean(np.log(clipped))

has_logits = all(column in predictions.columns for column in OPTIONAL_LOGIT_COLUMNS)
temperature = None
temperature_note = "Skipped temperature scaling because H10 did not provide safe_logit,suspicious_logit,scam_logit."
score_df = predictions.copy()

if has_logits:
    for column in OPTIONAL_LOGIT_COLUMNS:
        score_df[column] = pd.to_numeric(score_df[column], errors="coerce")
    score_df = score_df.dropna(subset=OPTIONAL_LOGIT_COLUMNS).copy()
    validation_df = score_df[score_df["split"].isin(["validation", "val", "dev"])].copy()
    if validation_df.empty:
        temperature_note = "Skipped temperature scaling because logits were present but no validation split was available."
    else:
        val_logits = validation_df[OPTIONAL_LOGIT_COLUMNS].to_numpy(dtype=float)
        val_labels = validation_df["true_id"].to_numpy(dtype=int)
        result = minimize_scalar(
            lambda t: negative_log_likelihood(val_logits, val_labels, t),
            bounds=(0.05, 10.0),
            method="bounded",
        )
        temperature = float(result.x)
        calibrated = softmax(score_df[OPTIONAL_LOGIT_COLUMNS].to_numpy(dtype=float) / temperature)
        score_df[["safe_prob", "suspicious_prob", "scam_prob"]] = calibrated
        temperature_note = f"Fit temperature={temperature:.4f} on {len(validation_df)} validation rows."

print(temperature_note)


## Threshold Sweep

Runtime mapping:

```python
scam if scam_prob >= tau_high
suspicious if tau_low <= scam_prob < tau_high
safe otherwise
```

Cost function: `1 * false_negative + 0.2 * false_positive`. For this sweep, a false negative is a true scam predicted `safe`; a false positive is a non-scam predicted `scam`.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

def threshold_verdicts(scam_prob, tau_low, tau_high):
    scam_prob = np.asarray(scam_prob, dtype=float)
    verdicts = np.full(scam_prob.shape, "safe", dtype=object)
    verdicts[(scam_prob >= tau_low) & (scam_prob < tau_high)] = "suspicious"
    verdicts[scam_prob >= tau_high] = "scam"
    return verdicts

def metric_row(df, tau_low, tau_high):
    y_true = df["true_verdict_norm"].to_numpy()
    y_pred = threshold_verdicts(df["scam_prob"].to_numpy(), tau_low, tau_high)
    false_negative = int(((y_true == "scam") & (y_pred == "safe")).sum())
    false_positive = int(((y_true != "scam") & (y_pred == "scam")).sum())
    cost = false_negative + 0.2 * false_positive
    return {
        "tau_low": float(tau_low),
        "tau_high": float(tau_high),
        "cost": float(cost),
        "false_negative": false_negative,
        "false_positive": false_positive,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)),
        "scam_precision": float(precision_score(y_true, y_pred, labels=["scam"], average="macro", zero_division=0)),
        "scam_recall": float(recall_score(y_true, y_pred, labels=["scam"], average="macro", zero_division=0)),
        "n": int(len(df)),
    }

candidate_splits = ["validation", "val", "dev", "heldout", "test"]
heldout_df = score_df[score_df["split"].isin(candidate_splits)].copy()
if heldout_df.empty:
    heldout_df = score_df[~score_df["split"].isin(["train", "training"])].copy()
if heldout_df.empty:
    heldout_df = score_df.copy()

taus = np.round(np.arange(0.05, 1.00, 0.05), 2)
rows = []
for tau_low in taus:
    for tau_high in taus:
        if tau_low < tau_high:
            rows.append(metric_row(heldout_df, tau_low, tau_high))

sweep = pd.DataFrame(rows).sort_values(
    ["cost", "macro_f1", "scam_recall", "accuracy", "tau_high", "tau_low"],
    ascending=[True, False, False, False, True, True],
).reset_index(drop=True)
best = sweep.iloc[0].to_dict()
best_pred = threshold_verdicts(heldout_df["scam_prob"].to_numpy(), best["tau_low"], best["tau_high"])
best_cm = confusion_matrix(heldout_df["true_verdict_norm"], best_pred, labels=LABELS)

sweep.to_csv(SWEEP_RESULTS_PATH, index=False)

print("heldout rows", len(heldout_df))
print("best", best)
print(pd.DataFrame(best_cm, index=[f"true_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS]))
print("saved", SWEEP_RESULTS_PATH)
sweep.head(10)


## Reliability Diagram

This plots H10/H12 scam probability against observed scam frequency. The selected thresholds are overlaid as vertical lines.


In [ ]:
import matplotlib.pyplot as plt

def reliability_bins(df, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    work = df.copy()
    work["bin"] = pd.cut(work["scam_prob"], bins=bins, include_lowest=True, right=True)
    grouped = work.groupby("bin", observed=False)
    out = grouped.agg(
        mean_predicted_scam_prob=("scam_prob", "mean"),
        observed_scam_frequency=("true_verdict_norm", lambda s: float((s == "scam").mean())),
        n=("id", "count"),
    ).reset_index()
    return out.dropna(subset=["mean_predicted_scam_prob", "observed_scam_frequency"])

reliability = reliability_bins(heldout_df, n_bins=10)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="0.6", label="perfect calibration")
ax.plot(
    reliability["mean_predicted_scam_prob"],
    reliability["observed_scam_frequency"],
    marker="o",
    label="H10 scam probability",
)
ax.axvline(best["tau_low"], color="#1f77b4", linestyle=":", label=f"tau_low={best['tau_low']:.2f}")
ax.axvline(best["tau_high"], color="#d62728", linestyle=":", label=f"tau_high={best['tau_high']:.2f}")
ax.set_xlabel("Predicted scam probability")
ax.set_ylabel("Observed scam frequency")
ax.set_title("H12 reliability diagram")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(RELIABILITY_PLOT_PATH, dpi=160)
plt.show()

print("saved", RELIABILITY_PLOT_PATH)
reliability


## Decision Output

Save artifacts for runtime integration and later review. Results are marked provisional when they come from `notebooks` paths rather than official H10 outputs.


In [ ]:
import json
from datetime import datetime, timezone

provisional = "notebooks" in str(H10_ARTIFACT_PATH)
status = "provisional" if provisional else "official-candidate"
cm_df = pd.DataFrame(best_cm, index=[f"true_{x}" for x in LABELS], columns=[f"pred_{x}" for x in LABELS])

metrics_payload = {
    "task": "H12",
    "status": status,
    "source_artifact": str(H10_ARTIFACT_PATH),
    "rows_evaluated": int(len(heldout_df)),
    "splits_evaluated": sorted(heldout_df["split"].unique().tolist()),
    "tau_low": float(best["tau_low"]),
    "tau_high": float(best["tau_high"]),
    "temperature": temperature,
    "temperature_note": temperature_note,
    "cost": float(best["cost"]),
    "false_negative": int(best["false_negative"]),
    "false_positive": int(best["false_positive"]),
    "accuracy": float(best["accuracy"]),
    "macro_f1": float(best["macro_f1"]),
    "scam_precision": float(best["scam_precision"]),
    "scam_recall": float(best["scam_recall"]),
    "confusion_matrix_labels": LABELS,
    "confusion_matrix": best_cm.tolist(),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

runtime_constants = {
    "gemscan_thresholds_version": "h12-v0",
    "status": status,
    "tau_low": float(best["tau_low"]),
    "tau_high": float(best["tau_high"]),
    "temperature": temperature,
    "verdict_rule": {
        "safe": "scam_prob < tau_low",
        "suspicious": "tau_low <= scam_prob < tau_high",
        "scam": "scam_prob >= tau_high",
    },
    "source_artifact": str(H10_ARTIFACT_PATH),
    "generated_at": metrics_payload["generated_at"],
}

BEST_METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2) + "\n")
RUNTIME_CONSTANTS_PATH.write_text(json.dumps(runtime_constants, indent=2) + "\n")

summary = f"""# H12 Threshold Sweep Decision

Status: **{status}**

Source artifact: `{H10_ARTIFACT_PATH}`

Selected thresholds:

- `tau_low`: `{best['tau_low']:.2f}`
- `tau_high`: `{best['tau_high']:.2f}`
- `temperature`: `{temperature if temperature is not None else 'not fit'}`

Temperature scaling: {temperature_note}

Held-out rows evaluated: `{len(heldout_df)}`

Metrics at selected threshold pair:

- Cost (`1 * FN + 0.2 * FP`): `{best['cost']:.4f}`
- False negatives: `{int(best['false_negative'])}`
- False positives: `{int(best['false_positive'])}`
- Accuracy: `{best['accuracy']:.4f}`
- Macro-F1: `{best['macro_f1']:.4f}`
- Scam precision: `{best['scam_precision']:.4f}`
- Scam recall: `{best['scam_recall']:.4f}`

Confusion matrix labels: `{LABELS}`

```text
{cm_df.to_string()}
```

Artifacts:

- Sweep CSV: `{SWEEP_RESULTS_PATH}`
- Reliability diagram: `{RELIABILITY_PLOT_PATH}`
- Metrics JSON: `{BEST_METRICS_PATH}`
- Runtime constants JSON: `{RUNTIME_CONSTANTS_PATH}`

Provisional note: results produced from `notebooks` paths are scaffolding outputs and must be rerun against official H10 predictions before runtime constants are baked into Swift/agent code.
"""

DECISION_SUMMARY_PATH.write_text(summary)

print("saved", BEST_METRICS_PATH)
print("saved", RUNTIME_CONSTANTS_PATH)
print("saved", DECISION_SUMMARY_PATH)
print(summary)
